In [22]:
import pandas as pd

life = pd.read_csv("../data/raw/life_expectancy.csv", skiprows=4)
print(life.shape)
life.head()

(266, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,64.049000,64.215000,64.602000,64.944000,65.303000,65.615000,...,75.620000,75.880000,76.019000,75.406000,73.655000,76.226000,76.353000,76.500000,NaN,NaN
1,Africa Eastern and Southern,AFE,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,44.169658,44.468838,44.877890,45.160583,45.535695,45.770723,...,62.591275,63.330691,63.857261,63.766484,62.979999,64.487152,65.146228,65.349930,NaN,NaN
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,32.799000,33.291000,33.757000,34.201000,34.673000,35.124000,...,62.406000,62.443000,62.941000,61.454000,60.417000,65.617000,66.035000,66.289000,NaN,NaN
3,Africa Western and Central,AFW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.779636,38.058973,38.681792,38.936918,39.194375,39.478875,...,56.620764,57.030771,57.142827,57.357709,57.355015,57.978995,58.847483,59.041538,NaN,NaN
4,Angola,AGO,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.933000,36.902000,37.168000,37.419000,37.704000,37.968000,...,62.122000,62.622000,63.051000,63.116000,62.958000,64.246000,64.617000,64.805000,NaN,NaN


In [23]:
meta = pd.read_csv("../data/raw/country_metadata.csv")
print(meta.columns.tolist())


['Country Code', 'Region', 'IncomeGroup', 'SpecialNotes', 'TableName', 'Unnamed: 5']


In [24]:
print(meta.shape)
meta.columns.tolist()

(265, 6)


['Country Code',
 'Region',
 'IncomeGroup',
 'SpecialNotes',
 'TableName',
 'Unnamed: 5']

In [25]:
meta["Region"].value_counts(dropna=False)

Region
Europe & Central Asia         58
NaN                           48
Sub-Saharan Africa            48
Latin America & Caribbean     42
East Asia & Pacific           37
Middle East & North Africa    23
South Asia                     6
North America                  3
Name: count, dtype: int64

### Cell 1: drop the junk columns

In [26]:
life = life.loc[:, ~life.columns.str.startswith("Unnamed")]
meta = meta.loc[:, ~meta.columns.str.startswith("Unnamed")]

print(life.shape, meta.shape)

(266, 70) (265, 5)


### Cell 2: find the mismatched row

In [27]:
merged = life.merge(meta, on="Country Code", how="outer", indicator=True)
print(merged["_merge"].value_counts())
merged[merged["_merge"] != "both"][["Country Name", "Country Code", "_merge"]]

_merge
both          265
left_only       1
right_only      0
Name: count, dtype: int64


,Country Name,Country Code,_merge
110,Not classified,INX,left_only


### Cell 3: keep Sub-Saharan Africa only

In [28]:
africa = merged[merged["Region"] == "Sub-Saharan Africa"].copy()
print(africa.shape)
africa[["Country Name", "Country Code", "2019"]].head(10)

(48, 75)


,Country Name,Country Code,2019
4,Angola,AGO,63.051
16,Burundi,BDI,62.180
18,Benin,BEN,59.888
19,Burkina Faso,BFA,60.184
33,Botswana,BWA,67.167
34,Central African Republic,CAF,31.530
41,Cote d'Ivoire,CIV,60.274
42,Cameroon,CMR,61.695
43,"Congo, Dem. Rep.",COD,60.242
44,"Congo, Rep.",COG,63.152


### Cell 4: check missing values and find Kenya

In [29]:
print("Missing in 2019:", africa["2019"].isna().sum())
africa[africa["Country Name"] == "Kenya"][["Country Name", "2019"]]

Missing in 2019: 0


,Country Name,2019
121,Kenya,62.939


In [37]:
year_cols = [str(y) for y in range(2012, 2025)]
caf = africa[africa["Country Code"] == "CAF"][["Country Name"] + year_cols]
caf.T

,34
Country Name,Central African Republic
2012,51.336
2013,49.487
2014,40.265
2015,51.934
2016,50.999
2017,45.231
2018,52.297
2019,31.53
2020,50.596


In [38]:
africa.sort_values("2019")[["Country Name", "2019"]].head(5)

,Country Name,2019
34,Central African Republic,31.530
229,Chad,52.993
174,Nigeria,53.009
141,Lesotho,55.249
213,"Somalia, Fed. Rep.",57.249
